06 Task-Aware Evaluation

Goal: evaluate pin quality beyond median offset. This notebook introduces movable-place evaluation, arrival-cost scoring, and optional comparison for future repositioning methods.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.features import place_complexity, pin_ambiguity, should_move_rule
from src.metrics import (
    haversine_meters,
    task_aware_report,
    segmented_task_report,
)

PROCESSED = PROJECT_ROOT / "data" / "processed"

PROJECT_ROOT


PosixPath('/Users/shivanibelambe/Pin-To-Place')

In [2]:
combined_path = PROCESSED / "ground_truth_combined.csv"

if combined_path.exists():
    df = pd.read_csv(combined_path)
else:
    files = sorted((PROCESSED / "ground_truth").glob("ground_truth_*.csv"))
    df = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)

df["place_complexity"] = df.get("place_complexity", df.apply(place_complexity, axis=1))
df["pin_ambiguity"] = df.get("pin_ambiguity", df.apply(pin_ambiguity, axis=1))
df["should_move"] = df.get("should_move", df.apply(should_move_rule, axis=1))

df.shape


(3425, 35)

In [3]:
baseline_report = task_aware_report(df)

baseline_report


{'count': 3418,
 'mean_m': np.float64(2.69),
 'median_m': np.float64(0.0),
 'p90_m': np.float64(20.01),
 'p95_m': np.float64(23.34),
 'max_m': np.float64(74.77),
 'pct_exact_no_move': np.float64(88.5),
 'pct_over_10m': np.float64(11.5),
 'pct_over_25m': np.float64(3.1),
 'pct_over_50m': np.float64(0.0)}

In [4]:
movable_df = df[df["should_move"]].copy()
protected_df = df[~df["should_move"]].copy()

{
    "all_rows": len(df),
    "movable_rows": len(movable_df),
    "protected_rows": len(protected_df),
    "movable_pct": round(len(movable_df) / len(df) * 100, 1),
}


{'all_rows': 3425,
 'movable_rows': 392,
 'protected_rows': 3033,
 'movable_pct': 11.4}

In [5]:
{
    "all_places": task_aware_report(df),
    "movable_places": task_aware_report(movable_df) if len(movable_df) else {},
    "protected_places": task_aware_report(protected_df) if len(protected_df) else {},
}


{'all_places': {'count': 3418,
  'mean_m': np.float64(2.69),
  'median_m': np.float64(0.0),
  'p90_m': np.float64(20.01),
  'p95_m': np.float64(23.34),
  'max_m': np.float64(74.77),
  'pct_exact_no_move': np.float64(88.5),
  'pct_over_10m': np.float64(11.5),
  'pct_over_25m': np.float64(3.1),
  'pct_over_50m': np.float64(0.0)},
 'movable_places': {'count': 392,
  'mean_m': np.float64(23.4),
  'median_m': np.float64(22.95),
  'p90_m': np.float64(27.63),
  'p95_m': np.float64(30.3),
  'max_m': np.float64(74.77),
  'pct_exact_no_move': np.float64(0.0),
  'pct_over_10m': np.float64(100.0),
  'pct_over_25m': np.float64(27.3),
  'pct_over_50m': np.float64(0.3)},
 'protected_places': {'count': 3026,
  'mean_m': np.float64(0.0),
  'median_m': np.float64(0.0),
  'p90_m': np.float64(0.0),
  'p95_m': np.float64(0.0),
  'max_m': np.float64(9.8),
  'pct_exact_no_move': np.float64(100.0),
  'pct_over_10m': np.float64(0.0),
  'pct_over_25m': np.float64(0.0),
  'pct_over_50m': np.float64(0.0)}}

In [6]:
segmented_task_report(df, "tier_label")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
3,2300,3.29,0.0,21.65,24.36,74.77,86.0,14.0,4.2,0.0,standard_commercial
2,209,7.34,0.0,23.13,24.33,32.98,67.5,32.5,4.8,0.0,open_space
0,163,0.43,0.0,0.00,0.00,26.97,98.2,1.8,0.6,0.0,multi_tenant
1,746,0.00,0.0,0.00,0.00,0.00,100.0,0.0,0.0,0.0,no_building


In [7]:
segmented_task_report(df, "place_complexity")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
1,122,3.66,0.0,22.09,25.03,31.87,84.4,15.6,5.7,0.0,multi_tenant
0,302,4.37,0.0,22.42,24.15,33.65,81.1,18.9,4.0,0.0,complex
2,2994,2.48,0.0,18.56,23.19,74.77,89.4,10.6,2.9,0.0,simple


In [8]:
segmented_task_report(df, "pin_ambiguity")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
1,2058,3.46,0.0,21.78,23.87,74.77,85.2,14.8,4.1,0.0,low
2,331,2.22,0.0,0.00,22.89,35.84,90.6,9.4,3.0,0.0,medium
0,1029,1.28,0.0,0.00,18.63,33.65,94.5,5.5,1.2,0.0,high


In [9]:
def infer_arrival_friction(row) -> dict:
    """
    V1 heuristic arrival-friction labels.

    These are intentionally conservative placeholders until you have
    sidewalk, curb-cut, road-network, parking-lot, or imagery-derived features.
    """
    tier = row.get("tier_label")
    complexity = row.get("place_complexity")
    category = str(row.get("category_primary", "")).lower()

    parking_lot_crossing = tier in {"standard_commercial", "multi_tenant"} and complexity in {
        "complex",
        "multi_tenant",
    }

    sidewalk_visible = None
    barrier_detected = False

    if tier == "open_space":
        sidewalk_visible = False

    if category in {"campground", "rv_park", "resort"}:
        parking_lot_crossing = True

    return {
        "sidewalk_visible": sidewalk_visible,
        "parking_lot_crossing": parking_lot_crossing,
        "barrier_detected": barrier_detected,
    }


friction = df.apply(infer_arrival_friction, axis=1, result_type="expand")
df = pd.concat([df, friction], axis=1)

df[["sidewalk_visible", "parking_lot_crossing", "barrier_detected"]].head()

,sidewalk_visible,parking_lot_crossing,barrier_detected
0,None,False,False
1,None,False,False
2,None,False,False
3,None,False,False
4,None,False,False


In [10]:
# Load OSM-based arrival cost (replaces the heuristic arrival_cost_score())
osm = pd.read_csv(PROCESSED / "osm_routing" / "osm_arrival_cost.csv")[["id", "osm_arrival_cost_m", "routing_penalty_m", "routing_method"]]
df = df.merge(osm, on="id", how="left")

# Use osm_arrival_cost_m as arrival_cost_m; fall back to haversine for any unmatched rows
df["arrival_cost_m"] = df["osm_arrival_cost_m"].fillna(df["offset_haversine_m"])

print(f"OSM-routed rows: {(df['routing_method']=='osrm').sum()}")
print(f"Zero-offset rows: {(df['routing_method']=='zero_offset').sum()}")
print(f"Unmatched (haversine fallback): {df['osm_arrival_cost_m'].isna().sum()}")
print()
task_aware_report(df, offset_col="arrival_cost_m")


OSM-routed rows: 365
Zero-offset rows: 3025
Unmatched (haversine fallback): 7



{'count': 3418,
 'mean_m': np.float64(11.18),
 'median_m': np.float64(0.0),
 'p90_m': np.float64(6.86),
 'p95_m': np.float64(25.7),
 'max_m': np.float64(6478.5),
 'pct_exact_no_move': np.float64(89.0),
 'pct_over_10m': np.float64(9.4),
 'pct_over_25m': np.float64(5.2),
 'pct_over_50m': np.float64(3.3)}

In [11]:
segmented_task_report(df, "tier_label", offset_col="arrival_cost_m")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
2,209,14.58,0.0,25.49,58.64,727.3,67.5,26.8,10.5,5.3,open_space
3,2300,15.17,0.0,14.40,42.11,6478.5,86.8,11.3,6.7,4.3,standard_commercial
0,163,1.64,0.0,0.00,0.00,209.3,98.2,1.8,1.2,0.6,multi_tenant
1,746,0.00,0.0,0.00,0.00,0.0,100.0,0.0,0.0,0.0,no_building


In [12]:
segmented_task_report(df, "place_complexity", offset_col="arrival_cost_m")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
1,122,18.83,0.0,18.00,89.43,537.5,85.2,12.3,9.0,6.6,multi_tenant
0,302,11.45,0.0,18.44,26.57,727.3,81.5,15.6,6.3,3.0,complex
2,2994,10.84,0.0,0.20,24.27,6478.5,89.9,8.6,4.9,3.2,simple


In [13]:
def compute_method_offset(df, lat_col, lon_col, output_col):
    result = df.copy()

    valid = result[[lat_col, lon_col, "gt_lat", "gt_lon"]].notna().all(axis=1)

    result[output_col] = np.nan
    result.loc[valid, output_col] = result.loc[valid].apply(
        lambda row: haversine_meters(
            row[lat_col],
            row[lon_col],
            row["gt_lat"],
            row["gt_lon"],
        ),
        axis=1,
    )

    return result


candidate_methods = {
    "ensemble": ("ensemble_lat", "ensemble_lon"),
    "ranker": ("ranker_lat", "ranker_lon"),
    "llm": ("llm_lat", "llm_lon"),
}

available_methods = {
    name: cols
    for name, cols in candidate_methods.items()
    if cols[0] in df.columns and cols[1] in df.columns
}

available_methods

{}

In [14]:
method_reports = []

for method_name, (lat_col, lon_col) in available_methods.items():
    offset_col = f"{method_name}_offset_m"
    df = compute_method_offset(df, lat_col, lon_col, offset_col)

    baseline = df["offset_haversine_m"]
    method = df[offset_col]

    valid = method.notna()
    regression_rate = round((method[valid] > baseline[valid]).mean() * 100, 2)

    report = task_aware_report(df[valid], offset_col=offset_col)
    report["method"] = method_name
    report["regression_rate_pct"] = regression_rate
    method_reports.append(report)

if method_reports:
    pd.DataFrame(method_reports).sort_values("p95_m")
else:
    "No repositioning method columns found yet. This is expected until ensemble/ranker/LLM outputs are generated."

In [15]:
evaluation_cols = [
    "id",
    "name",
    "category_primary",
    "region",
    "tier_label",
    "place_complexity",
    "pin_ambiguity",
    "should_move",
    "offset_haversine_m",
    "arrival_cost_m",
    "routing_penalty_m",
    "routing_method",
    "gt_confidence",
]

task_eval = df[[c for c in evaluation_cols if c in df.columns]].copy()
task_eval.to_csv(PROCESSED / "task_aware" / "task_aware_evaluation.csv", index=False)
task_eval.sort_values("arrival_cost_m", ascending=False).head(50)


,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,should_move,offset_haversine_m,arrival_cost_m,routing_penalty_m,routing_method,gt_confidence
2278,08f446c036375536037a6e82ce706414,Life Storage,self_storage_facility,TX,standard_commercial,simple,low,True,26.511476,6478.50,6451.99,osrm,0.9
2276,08f26c820b29a90603e92ad23d25e4ee,Raising Cane's,chicken_wings_restaurant,TX,standard_commercial,simple,low,True,25.593140,1877.20,1851.61,osrm,0.9
1451,08f489c1ad12598603da2ca1be91a273,Rincon Alegre,bar,TX,standard_commercial,simple,low,True,26.553433,1371.00,1344.45,osrm,0.9
3365,08f44cc51154ab0603168d49b5cad61e,Grand Reserve at Canton,accommodation,GA,standard_commercial,simple,low,True,28.025943,1073.40,1045.37,osrm,0.9
742,08f2a33d10b34d8a035e19b46e7fa75e,Olive Garden Italian Restaurant,salad_bar,MA,standard_commercial,simple,low,True,22.529941,859.10,836.57,osrm,0.9
1279,08f268cd962eccda034ade79a0e06fbd,Olive Garden Italian Restaurant,salad_bar,CO,standard_commercial,simple,low,True,23.444465,759.20,735.76,osrm,0.9
2303,08f441305256450603596e4dfefc80fb,Sun Resorts & Residences Ft. Myers Beach,campground,FL,open_space,complex,high,True,23.490443,727.30,703.81,osrm,0.8
1171,08f2a8a4f0c9620b03436f994cf66f79,La Quinta Inn & Suites by Wyndham Wytheville,resort,VA,standard_commercial,complex,high,True,24.362991,652.50,628.14,osrm,0.9
3260,08f26c80d819c9560334d52669860a72,Divine Audio Visual,home_theater_systems_stores,TX,standard_commercial,simple,low,True,22.177090,642.10,619.92,osrm,0.9
2113,08f48eb32939ac2103c112038e7160c3,Ahnala,american_restaurant,AZ,standard_commercial,simple,low,True,28.251072,639.40,611.15,osrm,0.9


In [16]:
summary_lines = []

summary_lines.append("Task-Aware Evaluation Summary")
summary_lines.append("")
summary_lines.append("Baseline geometric offset")
summary_lines.append(str(task_aware_report(df, offset_col="offset_haversine_m")))
summary_lines.append("")
summary_lines.append("Arrival-cost score (OSM-based via OSRM)")
summary_lines.append(str(task_aware_report(df, offset_col="arrival_cost_m")))
summary_lines.append("")
summary_lines.append("Arrival cost by tier")
summary_lines.append(segmented_task_report(df, "tier_label", offset_col="arrival_cost_m").to_string(index=False))
summary_lines.append("")
summary_lines.append("Arrival cost by complexity")
summary_lines.append(segmented_task_report(df, "place_complexity", offset_col="arrival_cost_m").to_string(index=False))

summary_text = "\n".join(summary_lines)
(PROCESSED / "task_aware" / "task_aware_evaluation_summary.txt").write_text(summary_text + "\n")
print(summary_text)


Task-Aware Evaluation Summary

Baseline geometric offset
{'count': 3418, 'mean_m': np.float64(2.69), 'median_m': np.float64(0.0), 'p90_m': np.float64(20.01), 'p95_m': np.float64(23.34), 'max_m': np.float64(74.77), 'pct_exact_no_move': np.float64(88.5), 'pct_over_10m': np.float64(11.5), 'pct_over_25m': np.float64(3.1), 'pct_over_50m': np.float64(0.0)}

Arrival-cost score (OSM-based via OSRM)
{'count': 3418, 'mean_m': np.float64(11.18), 'median_m': np.float64(0.0), 'p90_m': np.float64(6.86), 'p95_m': np.float64(25.7), 'max_m': np.float64(6478.5), 'pct_exact_no_move': np.float64(89.0), 'pct_over_10m': np.float64(9.4), 'pct_over_25m': np.float64(5.2), 'pct_over_50m': np.float64(3.3)}

Arrival cost by tier
 count  mean_m  median_m  p90_m  p95_m  max_m  pct_exact_no_move  pct_over_10m  pct_over_25m  pct_over_50m             segment
   209   14.58       0.0  25.49  58.64  727.3               67.5          26.8          10.5           5.3          open_space
  2300   15.17       0.0  14.40  42

First, place_complexity separates simple places from multi-tenant and complex places. This makes the analysis fairer because a campground, resort, or shopping center should not be judged the same way as a standalone pizza shop.

Second, pin_ambiguity marks whether a place has one obvious pin or several plausible targets. This gives the project a more research-like angle: sometimes the problem is not “bad coordinate,” it is “the coordinate is inherently ambiguous.”

Third, should_move turns repositioning into a conservative decision. Since your current median offset is already 0.0m, the model should first decide whether a pin deserves movement at all.

Fourth, arrival_cost_m starts moving the project beyond raw distance. It is a v1 score for practical arrival friction: not just “how far is the pin from the label,” but “does this pin make arrival harder?”